BGD - use the whole dataset to compute one update
for each iteration:
    use ALL data
    compute loss
    compute gradient (average)
    update weights once

SDG - use one data point at a time
for each iteration:
    for each sample:
        compute prediction
        compute error
        update weights immediately


QUESTION
- is sgd better? tried it with same code and params but got a better answer?
- why was the answer better?

In [36]:
import csv
import pandas as pd
from sklearn.model_selection import train_test_split
import numpy as np
from sklearn.preprocessing import StandardScaler

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

In [37]:
# read data

df = pd.read_csv('src/pima-indians-diabetes.data', skiprows=2, header=None)

X = df.iloc[:, :-1] # everything except last column
y = df.iloc[:, -1] # last col

In [38]:

# using state=42 makes result reproducable. remove for random
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.4,
    random_state=42
)

# scale the values
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# verify shape 18-12
print(X_train.shape)
print(X_test.shape)

(460, 8)
(308, 8)


In [39]:
y_train = y_train.values
y_test = y_test.values

X_train = np.hstack((np.ones((X_train.shape[0], 1)), X_train))
X_test = np.hstack((np.ones((X_test.shape[0], 1)), X_test))

print(y_train.shape)
print(X_train.shape)

(460,)
(460, 9)


In [40]:
# need a weight for all inputs BP Cholesterol Age Pregnant (w1,w2,w3,w4)
# need w0 (bias)
# score = w1*BP + w2*Cholesterol + w3*Age + w4*Pregnant + b

prev_loss = 0
current_loss = 0
threshold = 1e-6

fail_safe = 5000
iterations = 0
learning_rate = 0.01

epsilon = 1e-15 # const dont change

weights = np.zeros(X_train.shape[1])
while iterations < fail_safe:
    # one full pass through training set
    for i in range(len(X_train)):
        x_i = X_train[i]
        y_i = y_train[i]

        # 1. predict for one sample
        z = np.dot(x_i, weights)
        y_pred = sigmoid(z)

        # clip for numerical safety
        y_pred = np.clip(y_pred, epsilon, 1 - epsilon)

        # 2. compute error for one sample
        error = y_pred - y_i

        # 3. gradient for one sample
        gradient = x_i * error

        # 4. update immediately
        weights = weights - learning_rate * gradient

    # after one full pass, compute full loss for stopping
    z_all = np.dot(X_train, weights)
    y_pred_all = sigmoid(z_all)
    y_pred_all = np.clip(y_pred_all, epsilon, 1 - epsilon)

    m = len(y_train)
    current_loss = -(1/m) * np.sum(
        y_train * np.log(y_pred_all) + (1 - y_train) * np.log(1 - y_pred_all)
    )

    if iterations > 0 and abs(current_loss - prev_loss) < threshold:
        print(f"Iterations: {iterations} | current_loss: {current_loss}")
        break

    prev_loss = current_loss
    iterations += 1

Iterations: 13 | current_loss: 0.47559311869045146


In [41]:
# run the algo again but on test data
z_test = np.dot(X_test, weights)
y_prob = sigmoid(z_test)

# change probability into class label. Threshold is 0.5
y_pred = (y_prob >= 0.5).astype(int)

# compare accuracy
accuracy = np.mean(y_pred == y_test)

print(f"Accurary: {accuracy}")

Accurary: 0.7694805194805194


In [42]:
print("weights:")

for i, w in enumerate(weights):
    print(f"    w{i}: {w}")

weights:
    w0: -0.8094431671215209
    w1: 0.2363617487716034
    w2: 1.024062432658205
    w3: -0.21651322629296865
    w4: -0.061162361259438955
    w5: -0.01817214309551855
    w6: 0.8520976247190793
    w7: 0.14542473531822073
    w8: 0.36795230475630875
